# Blok 1. Seed i losowość w generowaniu obrazów

Notebook do pokazu trenera (blok 1, 0:15–0:45). Pokazuje, **dlaczego ten sam prompt daje różne obrazy** i jak seed pozwala iterować w kontrolowany sposób.

**Model:** FLUX.1 [schnell] przez API fal.ai (szybki, tani, licencja Apache 2.0, dopuszcza użycie komercyjne).

**Koszt:** 0,003 USD za megapiksel, czyli ok. 0,003 USD za obraz 1024×1024. Cały notebook (ok. 20 obrazów w niskiej rozdzielczości) kosztuje mniej niż 0,10 USD. Opcjonalny eksperyment 4 z modelem FLUX.1 [dev] (0,025 USD/MP) dokłada ok. 0,10 USD.

**Wymagania:**
- Python 3.10+, Jupyter (lokalnie) lub Google Colab (darmowy)
- konto na [fal.ai](https://fal.ai) i klucz API (Dashboard → Keys), doładowanie min. kilku USD
- pakiety: `fal-client`, `requests`, `pillow`

**Tryb demo:** jeśli nie podasz klucza, notebook działa w trybie `DEMO` i rysuje zastępcze obrazy. Przydaje się do sprawdzenia przed zajęciami lub gdy nie ma sieci. Prawdziwe wyniki zapisuj w folderze `wyniki/` przed zajęciami.

## 0. Instalacja i konfiguracja

In [ ]:
# Odkomentuj przy pierwszym uruchomieniu (lokalnie lub w Colab)
# !pip install -q fal-client requests pillow

In [ ]:
import os, io, time, getpass
from pathlib import Path

import requests
from PIL import Image, ImageDraw
from IPython.display import display

# Klucz API: zmienna środowiskowa FAL_KEY albo wpisanie poniżej.
# Nigdy nie wklejaj klucza na stałe do notebooka, który udostępniasz uczestnikom.
if not os.environ.get("FAL_KEY"):
    try:
        key = getpass.getpass("Klucz fal.ai (Enter = tryb DEMO): ")
    except Exception:
        key = ""
    if key.strip():
        os.environ["FAL_KEY"] = key.strip()

DEMO = not bool(os.environ.get("FAL_KEY"))
OUT = Path("wyniki"); OUT.mkdir(exist_ok=True)

# Licznik kosztów (USD za megapiksel, ceny fal.ai, październik 2026)
PRICE_PER_MP = {"fal-ai/flux/schnell": 0.003, "fal-ai/flux/dev": 0.025}
spent = {"usd": 0.0, "images": 0}

print("Tryb:", "DEMO (obrazy zastępcze, 0 USD)" if DEMO else "API fal.ai")

## 1. Funkcje pomocnicze

`generate()` wysyła prompt do modelu i zwraca obraz. Kluczowe parametry:

- `seed` – liczba, z której powstaje startowy szum; ten sam seed + prompt + ustawienia = ten sam obraz
- `steps` – liczba kroków odszumiania (schnell: 1–4, dev: ok. 28)
- `size` – rozmiar; do pokazów wystarcza 512×640 (taniej i szybciej)

In [ ]:
def _placeholder(prompt, seed, w, h):
    """Obraz zastępczy dla trybu DEMO: kolor zależny od seeda + podpis."""
    import random
    rnd = random.Random(f"{prompt}|{seed}")
    bg = tuple(rnd.randint(40, 200) for _ in range(3))
    img = Image.new("RGB", (w, h), bg)
    d = ImageDraw.Draw(img)
    for _ in range(6):
        x0, y0 = rnd.randint(0, w), rnd.randint(0, h)
        r = rnd.randint(20, w // 3)
        col = tuple(min(255, c + rnd.randint(-60, 60)) for c in bg)
        d.ellipse([x0 - r, y0 - r, x0 + r, y0 + r], fill=col)
    d.rectangle([0, h - 40, w, h], fill=(0, 0, 0))
    d.text((10, h - 30), f"DEMO seed={seed}", fill=(255, 255, 255))
    return img


def generate(prompt, seed=None, model="fal-ai/flux/schnell", size=(512, 640), steps=None):
    w, h = size
    if DEMO:
        return _placeholder(prompt, seed, w, h)
    import fal_client
    args = {
        "prompt": prompt,
        "image_size": {"width": w, "height": h},
        "num_images": 1,
        "enable_safety_checker": True,
    }
    if seed is not None:
        args["seed"] = int(seed)
    if steps is not None:
        args["num_inference_steps"] = steps
    result = fal_client.subscribe(model, arguments=args)
    url = result["images"][0]["url"]
    img = Image.open(io.BytesIO(requests.get(url, timeout=60).content)).convert("RGB")
    mp = max(1, -(-(w * h) // 1_000_000))  # fal zaokrągla w górę do pełnego MP
    spent["usd"] += mp * PRICE_PER_MP.get(model, 0)
    spent["images"] += 1
    return img


def grid(images, labels, cols=4, name=None):
    """Skleja obrazy w siatkę z podpisami i (opcjonalnie) zapisuje do wyniki/."""
    w, h = images[0].size
    rows = -(-len(images) // cols)
    pad = 28
    sheet = Image.new("RGB", (cols * w, rows * (h + pad)), (255, 255, 255))
    d = ImageDraw.Draw(sheet)
    for i, (im, lab) in enumerate(zip(images, labels)):
        x, y = (i % cols) * w, (i // cols) * (h + pad)
        sheet.paste(im.resize((w, h)), (x, y))
        d.text((x + 6, y + h + 6), str(lab), fill=(0, 0, 0))
    if name:
        sheet.save(OUT / f"{name}.jpg", quality=90)
    display(sheet)
    return sheet


def cost():
    print(f"Wygenerowano: {spent['images']} obrazów, koszt ok. {spent['usd']:.3f} USD")

## 2. Prompt bazowy

Prompt z moodboardu NORDA (biblioteka promptów, P1.1). Zwróć uwagę, czego **nie** określa: miejsca kamery, planu, obiektywu, kompozycji. O tym zdecyduje los.

In [ ]:
BASE = (
    "minimalist scandinavian coffee brand moodboard, navy blue and sand beige palette, "
    "winter baltic sea coast at dawn, soft natural light, film grain, editorial photography, calm and warm"
)
print(BASE)

## Eksperyment 1. Ten sam prompt, różne seedy

**Pytanie do grupy:** który z czterech obrazów jest „poprawny”?

**Wniosek:** wszystkie, bo prompt ich nie rozróżnia. Im mniej decyzji w prompcie, tym więcej decyduje losowość.

In [ ]:
seeds = [11, 42, 777, 2026]
imgs = [generate(BASE, seed=s) for s in seeds]
grid(imgs, [f"seed {s}" for s in seeds], name="e1_rozne_seedy")
cost()

## Eksperyment 2. Ten sam seed – powtarzalność

Generujemy dwa razy z seedem 42. Obrazy powinny być identyczne lub prawie identyczne (drobne różnice mogą wynikać z obliczeń na GPU).

**Wniosek:** seed + prompt + ustawienia = przepis, który można odtworzyć. W pracy z klientem zapisuj seed zaakceptowanej wersji.

In [ ]:
a = generate(BASE, seed=42)
b = generate(BASE, seed=42)
grid([a, b], ["seed 42 (1)", "seed 42 (2)"], cols=2, name="e2_powtarzalnosc")
cost()

## Eksperyment 3. Stały seed, zmiana jednego elementu

Zmieniamy tylko światło. Seed zostaje ten sam, więc kompozycja w dużej mierze się utrzymuje, a zmienia się to, o co prosimy.

**Wniosek:** kontrolowane iterowanie = stały seed + jedna zmiana naraz. Tak pracuje się nad wariantami dla klienta.

*Uwaga: przy dużych zmianach treści promptu kompozycja może się zmienić mimo stałego seeda.*

In [ ]:
LIGHTS = {
    "miękkie, pochmurne": "soft overcast light",
    "złota godzina": "low golden hour sun, long shadows",
    "niebieska godzina": "blue hour, warm interior lights glowing",
    "kontra": "strong backlight, rim light, silhouette",
}
base_no_light = BASE.replace("soft natural light", "{light}")
imgs = [generate(base_no_light.format(light=v), seed=42) for v in LIGHTS.values()]
grid(imgs, list(LIGHTS.keys()), name="e3_swiatlo")
cost()

## Eksperyment 4. Prompt fotografa vs prompt przymiotników

Porównujemy dwa podejścia przy tym samym seedzie:

- **A – przymiotniki:** „piękny, niesamowity, 8k, best quality”
- **B – opis kadru:** plan, kąt, obiektyw, światło, kompozycja

**Wniosek:** prompt B daje przewidywalny kadr, A daje „ładny przypadek”.

In [ ]:
PROMPT_A = "beautiful amazing coffee can, stunning, masterpiece, 8k, best quality, ultra detailed"
PROMPT_B = (
    "navy blue aluminium coffee can with minimal sand-coloured label on a frosted wooden pier, "
    "calm grey baltic sea in background, overcast winter morning soft light, low angle close-up, "
    "85mm lens, shallow depth of field, negative space on the left, editorial product photography"
)
seeds = [7, 42]
imgs, labels = [], []
for s in seeds:
    imgs += [generate(PROMPT_A, seed=s), generate(PROMPT_B, seed=s)]
    labels += [f"A przymiotniki, seed {s}", f"B opis kadru, seed {s}"]
grid(imgs, labels, cols=4, name="e4_przymiotniki_vs_kadr")
cost()

## Eksperyment 5 (opcjonalny). Szybki model vs dokładniejszy

FLUX.1 [schnell] (4 kroki, 0,003 USD/MP) vs FLUX.1 [dev] (28 kroków, 0,025 USD/MP, ok. 8× drożej).

**Wniosek do omówienia:** szkice i eksploracja na tanim modelu, finał na dokładniejszym. Ta sama zasada działa w Midjourney (Relax/Fast), Flow (Fast/Quality) i Kling.

*Licencja: FLUX.1 [dev] ma licencję niekomercyjną dla samych wag; użycie przez API fal.ai podlega warunkom fal.ai. Przed użyciem w projekcie klienta sprawdź aktualne warunki.*

In [ ]:
RUN_DEV = False  # zmień na True, aby uruchomić (ok. 0,10 USD)

if RUN_DEV:
    s = 42
    fast = generate(PROMPT_B, seed=s, model="fal-ai/flux/schnell", steps=4)
    slow = generate(PROMPT_B, seed=s, model="fal-ai/flux/dev", steps=28)
    grid([fast, slow], ["schnell, 4 kroki", "dev, 28 kroków"], cols=2, name="e5_schnell_vs_dev")
    cost()
else:
    print("Pominięto. Ustaw RUN_DEV = True, aby uruchomić.")

## Podsumowanie dla grupy

1. Generowanie startuje z losowego szumu – seed ustala ten start.
2. Ten sam prompt bez seeda = różne obrazy. To nie błąd, tylko sposób działania.
3. Stały seed + jedna zmiana naraz = kontrolowane iterowanie.
4. Opis kadru (plan, kąt, obiektyw, światło) działa lepiej niż przymiotniki.
5. Szkicuj tanio, finalizuj dokładnie.

W Midjourney i Flow seed jest ukryty lub działa ograniczenie – dlatego w praktyce kontrolę dają referencje stylu i obrazu (blok 2).

**Źródła cen:** [fal.ai – FLUX.1 schnell](https://fal.ai/models/fal-ai/flux/schnell), [fal.ai – FLUX.1 dev](https://fal.ai/models/fal-ai/flux/dev) (sprawdzone październik 2026).

In [ ]:
cost()
print('Pliki w:', OUT.resolve())